In [ ]:
import pandas as pd
import numpy as np

# 1. 자치구 리스트 및 코드 매핑 정의
seoul_gu_list = [
    '종로구', '중구', '용산구', '성동구', '광진구', '동대문구', '중랑구', '성북구', '강북구', '도봉구',
    '노원구', '은평구', '서대문구', '마포구', '양천구', '강서구', '구로구', '금천구', '영등포구', '동작구',
    '관악구', '서초구', '강남구', '송파구', '강동구'
]

# 행정표준코드(5자리) 매핑
seoul_gu_map = {
    11110: '종로구', 11140: '중구', 11170: '용산구', 11200: '성동구', 11215: '광진구',
    11230: '동대문구', 11260: '중랑구', 11290: '성북구', 11305: '강북구', 11320: '도봉구',
    11350: '노원구', 11380: '은평구', 11410: '서대문구', 11440: '마포구', 11470: '양천구',
    11500: '강서구', 11530: '구로구', 11545: '금천구', 11560: '영등포구', 11590: '동작구',
    11620: '관악구', 11650: '서초구', 11680: '강남구', 11710: '송파구', 11740: '강동구'
}

# KT 교통 데이터(4자리) 매핑
b075_gu_map = {1101+i: v for i, v in enumerate(seoul_gu_list)}

# 2. 데이터 로드 함수
def load_data(file_name):
    for enc in ['utf-8-sig', 'utf-8', 'cp949']:
        try:
            return pd.read_csv(file_name, encoding=enc)
        except:
            continue
    return None

# 파일 로드 (파일명 확인 필수)
df_mobility = load_data('20191001_OUTPUT.csv')      # B075 (교통)
df_sales = load_data('행정동별 추정매출액.csv')       # B058 (경제)
df_price = load_data('예측시세.csv')                 # B019 (주거)
df_spatial = load_data('도로명주소 기반 건물공간정보.csv') # B028 (공간)

# 3. 데이터 집계 (Aggregation)
# A. 교통(T): 유입 인구 합산
df_mobility['자치구'] = df_mobility['도착지 코드(arv_place_cd)'].astype(str).str[:4].astype(int).map(b075_gu_map)
sum_t = df_mobility.groupby('자치구')['인구수(popl_cnt)'].sum().reset_index()

# B. 경제(B): 매출액 합산
df_sales['자치구'] = df_sales['행정동코드(ADSTRD_CD)'].astype(str).str[:5].astype(int).map(seoul_gu_map)
sum_b = df_sales.groupby('자치구')['매출액_합계(SUM_SELNG_AMT)'].sum().reset_index()

# C. 주거(R): m2당 시세 평균
df_price['자치구'] = df_price['법정동_구코드(SREG)'].map(seoul_gu_map)
df_price['m2_price'] = df_price['현재월예측시세(CENTERVALUE)'] / df_price['전용면적(JYAREA)']
sum_r = df_price.groupby('자치구')['m2_price'].mean().reset_index()

# D. 공간(S): 지상 층수 평균
df_spatial['자치구'] = df_spatial['시군구_코드(SIGNGU_CD)'].map(seoul_gu_map)
sum_s = df_spatial.groupby('자치구')['지상_층_수(GROUND_FLOOR_CO)'].mean().reset_index()

# 4. 데이터 통합 및 결측치 보정
master_df = pd.DataFrame({'자치구': seoul_gu_list})
master_df = master_df.merge(sum_t, on='자치구', how='left').merge(sum_b, on='자치구', how='left') \
                     .merge(sum_r, on='자치구', how='left').merge(sum_s, on='자치구', how='left')

# 결측값 중앙값 보정 (전체 구 표시용)
master_df = master_df.fillna(master_df.median(numeric_only=True))

# 5. 스코어링 로직 (0-100 정규화)
def norm(s):
    return (s - s.min()) / (s.max() - s.min()) * 100

# 개별 지표 점수화
master_df['T_Score'] = norm(master_df['인구수(popl_cnt)'])               # 교통 유입
master_df['B_Score'] = 100 - norm(master_df['매출액_합계(SUM_SELNG_AMT)']) # 경제 부족
master_df['R_Score'] = 100 - norm(master_df['m2_price'])               # 주거 저평가
master_df['S_Score'] = 100 - norm(master_df['지상_층_수(GROUND_FLOOR_CO)']) # 공간 여력

# 6. 가중치 적용 모델 (교통 40%, 나머지 각 20%)
master_df['Weighted_Score'] = (
    master_df['T_Score'] * 0.40 +
    master_df['B_Score'] * 0.20 +
    master_df['R_Score'] * 0.20 +
    master_df['S_Score'] * 0.20
).round(2)

# 7. 타게팅 예측 (Priority 설정)
def predict_target(score):
    if score >= 80: return 'High Priority'
    elif score >= 60: return 'Strategic Growth'
    else: return 'Monitoring'

# 결과 정렬 및 출력
final_result = master_df[['자치구', 'T_Score', 'B_Score', 'R_Score', 'S_Score', 'Weighted_Score']]
final_result = final_result.sort_values(by='Weighted_Score', ascending=False)

# 8. CSV 저장
final_result.to_csv('Seoul_District_Weighted_Score_Analysis.csv', index=False, encoding='utf-8-sig')

# 결과 출력
print(final_result.head(25))

     자치구     T_Score     B_Score     R_Score     S_Score  Weighted_Score
8    강북구  100.000000   99.610607   93.792596   12.679426           81.22
6    중랑구   77.589933   99.081107   77.150770   59.740260           78.23
4    광진구   34.409559   99.081107   97.025519   91.711230           71.33
10   노원구   38.960214  100.000000   93.286459   68.181818           67.88
13   마포구   25.753146   98.890783   86.704705  100.000000           67.42
12  서대문구   55.866277   60.899655   80.681446   83.732057           67.41
15   강서구   46.154824   99.081107   77.901078   67.045455           67.27
19   동작구   37.867040   93.484357   67.685528   91.590909           65.70
16   구로구   36.773866   98.026914   86.266740   60.427807           63.65
18  영등포구   37.867040   99.571574   92.086218   44.545455           62.39
14   양천구   54.518876   99.081107   97.201073    0.000000           61.06
24   강동구   37.867040   94.887119   80.667103   38.311688           57.92
20   관악구   37.867040   99.081107   78.463356   34.9

개별 스코어 (0~100점)
모든 점수는 100점에 가까울수록 '고밀 개발을 위한 잠재력(또는 필요성)'이 높다는 뜻입니다.

T_Score (Transport): 교통 활동성 점수

해당 구로 유입되는 인구량이 많을수록 점수가 높습니다.

의미: "이미 사람이 많이 모이는 곳이니, 이곳을 개발하면 효과가 가장 크다"는 인프라 효율성을 뜻합니다.

B_Score (Business): 업무 기능 결핍 점수

지역 내 매출액이 낮을수록 점수가 높습니다. (Inverse 처리)

의미: "유동인구에 비해 일자리가 부족하니, 업무 시설을 채워 넣어야 한다"는 뜻입니다.

R_Score (Residential): 주거 가치 저평가 점수

부동산 시세가 주변 입지에 비해 낮을수록 점수가 높습니다. (Inverse 처리)

의미: "교통은 좋은데 집값이 싸니, 고밀 주거 단지를 조성했을 때 가치 상승 폭이 크다"는 뜻입니다.

S_Score (Space): 공간 개발 여력 점수

현재 건물의 평균 층수가 낮을수록 점수가 높습니다. (Inverse 처리)

의미: "건물이 낮게 지어져 있으니, 위로 더 높게 올릴 수 있는 땅의 여유가 많다"는 물리적 잠재력입니다.

Weighted_Score (가중치 적용 종합 점수)

교통(T)에 40%, 나머지(B, R, S)에 각각 20%의 가중치를 주어 합산한 최종 점수입니다.

이 점수가 높을수록 오세훈 서울시장의 역세권 활성화 정책의 1순위 타겟이 됩니다.

In [ ]:
import pandas as pd
import numpy as np

# 1. 데이터 로드 함수 (인코딩 대응)
def load_data(file_name):
    for enc in ['utf-8-sig', 'utf-8', 'cp949']:
        try:
            return pd.read_csv(file_name, encoding=enc)
        except:
            continue
    return None

# 파일 로드 (B075, B058, B019, B028)
df_mobility = load_data('20191001_OUTPUT.csv')      # 교통
df_sales = load_data('행정동별 추정매출액.csv')       # 경제
df_price = load_data('예측시세.csv')                 # 주거
df_spatial = load_data('도로명주소 기반 건물공간정보.csv') # 공간

# 2. 서울시 25개 자치구 매핑 정의
seoul_gu_list = ['종로구', '중구', '용산구', '성동구', '광진구', '동대문구', '중랑구', '성북구', '강북구', '도봉구',
                 '노원구', '은평구', '서대문구', '마포구', '양천구', '강서구', '구로구', '금천구', '영등포구', '동작구',
                 '관악구', '서초구', '강남구', '송파구', '강동구']

# 자치구 코드 매핑 (5자리 행정표준코드)
seoul_gu_map = {11110: '종로구', 11140: '중구', 11170: '용산구', 11200: '성동구', 11215: '광진구',
                11230: '동대문구', 11260: '중랑구', 11290: '성북구', 11305: '강북구', 11320: '도봉구',
                11350: '노원구', 11380: '은평구', 11410: '서대문구', 11440: '마포구', 11470: '양천구',
                11500: '강서구', 11530: '구로구', 11545: '금천구', 11560: '영등포구', 11590: '동작구',
                11620: '관악구', 11650: '서초구', 11680: '강남구', 11710: '송파구', 11740: '강동구'}

# KT 유동인구용 4자리 코드 매핑
b075_gu_map = {1101+i: v for i, v in enumerate(seoul_gu_list)}

# 3. 데이터 집계
# 교통(T): 인구 유입 합산
df_mobility['자치구'] = df_mobility['도착지 코드(arv_place_cd)'].astype(str).str[:4].astype(int).map(b075_gu_map)
sum_t = df_mobility.groupby('자치구')['인구수(popl_cnt)'].sum().reset_index()

# 경제(B): 매출 규모 합산
df_sales['자치구'] = df_sales['행정동코드(ADSTRD_CD)'].astype(str).str[:5].astype(int).map(seoul_gu_map)
sum_b = df_sales.groupby('자치구')['매출액_합계(SUM_SELNG_AMT)'].sum().reset_index()

# 주거(R): 시세 집중도 평균
df_price['자치구'] = df_price['법정동_구코드(SREG)'].map(seoul_gu_map)
sum_r = df_price.groupby('자치구')['현재월예측시세(CENTERVALUE)'].mean().reset_index()

# 공간(S): 물리적 건물 밀도 평균
df_spatial['자치구'] = df_spatial['시군구_코드(SIGNGU_CD)'].map(seoul_gu_map)
sum_s = df_spatial.groupby('자치구')['지상_층_수(GROUND_FLOOR_CO)'].mean().reset_index()

# 4. 통합 및 정규화 기반 스코어링
master_df = pd.DataFrame({'자치구': seoul_gu_list})
master_df = master_df.merge(sum_t, on='자치구', how='left').merge(sum_b, on='자치구', how='left') \
                     .merge(sum_r, on='자치구', how='left').merge(sum_s, on='자치구', how='left')
master_df = master_df.fillna(master_df.median(numeric_only=True))

def norm(s): return (s - s.min()) / (s.max() - s.min()) * 100

master_df['T_Density'] = norm(master_df['인구수(popl_cnt)'])
master_df['B_Density'] = norm(master_df['매출액_합계(SUM_SELNG_AMT)'])
master_df['R_Density'] = norm(master_df['현재월예측시세(CENTERVALUE)'])
master_df['S_Density'] = norm(master_df['지상_층_수(GROUND_FLOOR_CO)'])

# 종합 과밀 스코어 산출
master_df['Overcrowding_Score'] = (
    master_df['T_Density'] * 0.40 +
    master_df['B_Density'] * 0.20 +
    master_df['R_Density'] * 0.20 +
    master_df['S_Density'] * 0.20
).round(2)

# 5. 모든 자치구 결과 정렬 출력
final_result = master_df[['자치구', 'T_Density', 'B_Density', 'R_Density', 'S_Density', 'Overcrowding_Score']]
final_result = final_result.sort_values(by='Overcrowding_Score', ascending=False)


# 인코딩 utf-8-sig는 한글 엑셀에서 바로 열어도 글자가 깨지지 않게 해줍니다.
output_file = 'Seoul_District_Overcrowding_Score.csv'
final_result.to_csv(output_file, index=False, encoding='utf-8-sig')

pd.set_option('display.max_rows', None)
print(final_result)

     자치구   T_Density   B_Density   R_Density   S_Density  Overcrowding_Score
8    강북구  100.000000    0.389393   14.370918   87.320574               60.42
7    성북구   45.875175  100.000000   37.540470   40.259740               53.91
14   양천구   54.518876    0.918893   19.191157  100.000000               45.83
21   서초구   37.867040   86.536756   17.107201   37.603306               43.40
6    중랑구   77.589933    0.918893   18.826467   40.259740               43.04
5   동대문구   51.442735    0.918893   54.588582   54.870130               42.65
23   송파구   37.867040    0.547371   28.092796   99.431818               40.76
3    성동구   27.507309    0.918893  100.000000   44.385027               40.06
22   강남구   37.867040    1.677174   35.450544   81.636364               38.90
12  서대문구   55.866277   39.100345   21.918205   16.267943               37.80
0    종로구   39.900852    0.472179    0.000000   97.107438               35.48
1     중구   35.121393    0.918893   14.981776   89.473684               35.12

종합 점수($Overcrowding\_Score$)(과밀스코어)는 다음과 같은 비중으로 합산되었습니다.$$Score = (T \times 0.4) + (B \times 0.2) + (R \times 0.2) + (S \times 0.2)$$교통(T): 40% (가장 핵심적인 과밀 요인), 경제(B), 주거(R), 공간(S): 각각 20%

① 교통 과밀 지수 ($T_{Density}$)원천 데이터: B075 (도착지 기준 유입 인구수 합계)의미: 해당 자치구로 유입되는 인구의 양이 많을수록 교통 혼잡 및 유동 인구에 의한 과밀 부하가 높다고 판단합니다. 식: $T_{Density} = \frac{X - min(T)}{max(T) - min(T)} \times 100$

② 경제 활동 밀도 ($B_{Density}$)원천 데이터: B058 (행정동별 추정 매출액 합계)의미: 상업 활동 및 매출 규모가 클수록 유동 인구의 집중도와 경제적 과열 정도가 높음을 나타냅니다. 식: $B_{Density} = \frac{X - min(B)}{max(B) - min(B)} \times 100$

③ 주거 수요 집중도 ($R_{Density}$)원천 데이터: B019 (예측 시세의 자치구별 평균값)의미: 시세가 높게 형성될수록 해당 지역에 대한 주거 수요가 집중되어 있으며, 인구 밀집 압력이 강함을 의미합니다. 식: $R_{Density} = \frac{X - min(R)}{max(R) - min(R)} \times 100$

④ 물리적 공간 밀도 ($S_{Density}$)원천 데이터: B028 (건물의 지상 층수 평균)의미: 건물의 평균 층수가 높을수록 이미 수직적 개발이 진행되어 물리적인 수용 한계에 도달한 과밀 상태로 간주합니다. 식: $S_{Density} = \frac{X - min(S)}{max(S) - min(S)} \times 100$

이 결과표는 정책 결정 시 다음과 같은 근거가 됩니다.

교통 대책 시급: 강북구, 중랑구처럼 T_Density가 높은 곳은 지하철 노선 증설이나 도로 확장이 우선되어야 합니다.

공급 억제 및 분산: 양천구, 송파구처럼 S_Density가 이미 100에 가까운 곳은 추가 개발보다는 인구를 주변으로 분산시키는 정책이 필요합니다.

직주근접 실패 지역: T_Density는 높은데 B_Density가 0에 가까운 곳(예: 강북구)은 전형적인 '베드타운'입니다. 이곳에 일자리를 만들어주면 이동 과밀을 해결할 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd

# 1. 쌍대 비교 행렬 (Pairwise Matrix)
# 기준: [교통, 매출, 시세, 공간]
A = np.array([
    [1,   2,   2,   2],   # 교통
    [1/2, 1,   1,   1],   # 매출
    [1/3, 1,   1,   1],   # 시세
    [1/2, 1,   1,   1]    # 공간
])

# 2. 고유값 계산
eigenvalues, eigenvectors = np.linalg.eig(A)

# 3. 최대 고유값 찾기
max_index = np.argmax(eigenvalues.real)
weights = eigenvectors[:, max_index].real

# 4. 정규화 (합이 1이 되도록)
weights = weights / weights.sum()

# 결과 출력
criteria = ['교통', '매출', '시세', '공간']
ahp_weights = pd.Series(weights, index=criteria)

print("AHP 가중치 결과")
print(ahp_weights)

AHP 가중치 결과
교통    0.406930
매출    0.203465
시세    0.186141
공간    0.203465
dtype: float64
